# CS 5326 PA3 — Search over Reasoning: full benchmark pipeline

Runs all six configurations on the 50-problem GSM8K benchmark and writes the required artifacts:

| File | Content |
|---|---|
| `outputs/per_problem.jsonl` | one line per problem: greedy output, all 15 candidates, step scores / rationales / raw judge outputs for the 5 shared candidates, every method's answer |
| `outputs/results_summary.json` | 6-row method table (accuracy, calls, tokens, time) |
| `outputs/aggregation_ablation.json` | min / product / step_penalty at λ ∈ {0, 0.5, 2}, with tie rates |
| `outputs/sc_scaling_curve.json` | self-consistency accuracy for N = 1 … 15 |

**Models:** generator `Qwen/Qwen2.5-1.5B-Instruct` (fp16, `cuda:0`), judge `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4, `cuda:1`).
**Seeds:** `transformers.set_seed(5326 + problem_index)` before each problem (0-based index).
**Resumable:** every finished problem is appended to `outputs/per_problem.jsonl` immediately; re-running skips finished problems.

All algorithmic code lives in `src/` (prompting, parsing, judge, aggregation, search, `src/pipeline.py` for the benchmark loop). This notebook only configures and calls it.

## 1. Configuration

In [1]:
PROBLEM_LIMIT = None        # None = all 50 problems. Set e.g. 3 for a quick dry run.
JUDGE_BATCH_SIZE = 16       # judge prompts per GPU batch (auto-halves on CUDA OOM)
COMPLIANCE_GATE = 80.0      # stop if < this % of greedy outputs have a parseable Final Answer (None = never stop)
FRESH_START = False         # True = delete previous checkpoints/outputs and start over
RESTORE_FROM_INPUT = True   # Kaggle: resume from checkpoints found in an attached previous notebook output
CODE_DATASET_HINT = "cs5326-pa3-code"   # name of the Kaggle dataset holding src/, data/, tests/, submission_adapter.py
RUN_LOGPROB_BONUS = True    # optional bonus: continuous step scores from judge logits ("product_logprob")
LOGPROB_BATCH_SIZE = 16     # judge prompts per forward-pass batch for the bonus (auto-halves on CUDA OOM)

import time
RUN_START = time.time()

## 2. Locate the code, set the working directory, restore checkpoints

In [2]:
import glob, json, os, shutil, sys

IN_KAGGLE = os.path.isdir("/kaggle/input")
CODE_ITEMS = ["src", "tests", "data", "submission_adapter.py"]

def find_code_root():
    # Running from the repository root (local Jupyter, or a git clone on Kaggle)
    if os.path.isfile("submission_adapter.py") and os.path.isdir("src"):
        return os.getcwd()
    hits = glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
    hits = [h for h in hits if "/outputs/" not in h]
    preferred = [h for h in hits if CODE_DATASET_HINT in h]
    hits = sorted(preferred or hits, key=len)
    if not hits:
        raise FileNotFoundError(
            "PA3 code not found. On Kaggle: right sidebar -> Add Input -> select your "
            f"'{CODE_DATASET_HINT}' dataset."
        )
    return os.path.dirname(hits[0])

code_root = find_code_root()
if code_root.startswith("/kaggle/input"):
    # /kaggle/input is read-only: copy the code to /kaggle/working/pa3 (outputs/ is kept between runs)
    work = "/kaggle/working/pa3"
    os.makedirs(work, exist_ok=True)
    for item in CODE_ITEMS:
        src_p, dst_p = os.path.join(code_root, item), os.path.join(work, item)
        if os.path.isdir(dst_p):
            shutil.rmtree(dst_p)
        elif os.path.isfile(dst_p):
            os.remove(dst_p)
        (shutil.copytree if os.path.isdir(src_p) else shutil.copy2)(src_p, dst_p)
    os.chdir(work)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Code from:        ", code_root)
print("Working directory:", os.getcwd())

os.makedirs("outputs", exist_ok=True)
CHECKPOINTS = ["per_problem.jsonl", "greedy_checkpoint.jsonl", "logprob_bonus.jsonl"]
RESTORED = []
if FRESH_START:
    for name in CHECKPOINTS + ["results_summary.json", "aggregation_ablation.json", "sc_scaling_curve.json",
                               "logprob_bonus_summary.json"]:
        p = os.path.join("outputs", name)
        if os.path.exists(p):
            os.remove(p)
            print("Deleted", p)
elif RESTORE_FROM_INPUT and IN_KAGGLE:
    for name in CHECKPOINTS:
        local = os.path.join("outputs", name)
        prev = sorted(glob.glob(f"/kaggle/input/**/outputs/{name}", recursive=True))
        if not os.path.exists(local) and prev:
            shutil.copy2(prev[0], local)
            RESTORED.append(prev[0])
            print(f"Restored checkpoint {name} from {prev[0]}")
for name in CHECKPOINTS:
    p = os.path.join("outputs", name)
    n = sum(1 for _ in open(p, encoding="utf-8")) if os.path.exists(p) else 0
    print(f"{p}: {n} problems already done")

Code from:         /kaggle/input/datasets/ibrahimhassan2424/cs5326-pa3-code/cs5326-pa3-code
Working directory: /kaggle/working/pa3
Restored checkpoint per_problem.jsonl from /kaggle/input/datasets/ibrahimhassan2424/pa3-checkpoints/outputs/per_problem.jsonl
Restored checkpoint greedy_checkpoint.jsonl from /kaggle/input/datasets/ibrahimhassan2424/pa3-checkpoints/outputs/greedy_checkpoint.jsonl
outputs/per_problem.jsonl: 50 problems already done
outputs/greedy_checkpoint.jsonl: 50 problems already done
outputs/logprob_bonus.jsonl: 0 problems already done


## 3. Dependencies (Kaggle already has torch / transformers / accelerate; only bitsandbytes must be upgraded)

In [3]:
import subprocess
if IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)

## 4. Public smoke tests (offline, mock models) — confirms the right code version is loaded

In [4]:
res = subprocess.run([sys.executable, "tests/test_public_adapter_smoke.py"], capture_output=True, text=True)
print(res.stdout[-1500:], res.stderr[-1500:])
assert res.returncode == 0, "Public smoke tests failed: wrong/old code version attached?"

Running 20 public adapter smoke tests...
  [PASS] test_format_step_prompt
  [PASS] test_parse_steps_plain
  [PASS] test_parse_steps_markdown_decorated
  [PASS] test_extract_answer_required_behaviors
  [PASS] test_extract_answer_resilience
  [PASS] test_compute_majority_vote
  [PASS] test_run_greedy
  [PASS] test_run_self_consistency
  [PASS] test_build_prm_judge_prompt_structure
  [PASS] test_parse_judge_decision
  [PASS] test_score_path
  [PASS] test_aggregate_path_score
  [PASS] test_select_best_path_clear_winner
  [PASS] test_select_best_path_tie_broken_by_majority
  [PASS] test_select_best_path_answer_none_exclusion
  [PASS] test_select_best_path_all_answers_none
  [PASS] test_run_prm_best_of_n_standalone
  [PASS] test_run_prm_best_of_n_with_shared_candidates
  [PASS] test_run_compute_matched_self_consistency
  [PASS] test_scaling_curve_consistency
Test Summary: 20 passed, 0 failed.
ALL TESTS PASSED SUCCESSFULLY!
 


## 5. Environment info (for report Section 5)

In [5]:
import platform
import torch, transformers, accelerate
try:
    import bitsandbytes
    bnb_version = bitsandbytes.__version__
except Exception as e:
    bnb_version = f"import failed: {e}"

assert torch.cuda.is_available(), "No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2"
ENV = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bnb_version,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "gpu_memory_gb": [round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
                      for i in range(torch.cuda.device_count())],
    "seed_protocol": "transformers.set_seed(5326 + problem_index), problem_index 0-based",
}
print(json.dumps(ENV, indent=2))

{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "transformers": "5.20.2",
  "accelerate": "1.15.0",
  "bitsandbytes": "0.50.2",
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ],
  "gpu_memory_gb": [
    15.6,
    15.6
  ],
  "seed_protocol": "transformers.set_seed(5326 + problem_index), problem_index 0-based"
}


## 6. Load both models

In [6]:
from src.utils import load_models, make_hf_generate_fn, make_hf_batch_judge_fn
from src.pipeline import CachedJudge

t0 = time.time()
gen_model, gen_tokenizer, judge_model, judge_tokenizer = load_models(generator_device="cuda:0", judge_device="cuda:1")
gen_model.eval(); judge_model.eval()
print(f"Models loaded in {time.time() - t0:.0f}s")
print("Generator device:", gen_model.device, "| Judge device:", judge_model.device)
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} allocated {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
ENV["vram_after_load_gb"] = {f"cuda:{i}": round(torch.cuda.memory_allocated(i) / 1e9, 2) for i in range(torch.cuda.device_count())}

generate_fn = make_hf_generate_fn(gen_model, gen_tokenizer)
judge = CachedJudge(batch_judge_fn=make_hf_batch_judge_fn(judge_model, judge_tokenizer), batch_size=JUDGE_BATCH_SIZE)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (11 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Models loaded in 153s
Generator device: cuda:0 | Judge device: cuda:1
cuda:0 allocated 3.09 GB
cuda:1 allocated 5.56 GB


## 7. Phase A — greedy on every problem + format-compliance check

The manual (§4.2) warns that Qwen2.5-1.5B can drift into markdown / LaTeX / `\boxed{}` and drop the `Final Answer:` line. Check the numbers below **before** the long run.

In [7]:
from tqdm.auto import tqdm
from src.pipeline import load_benchmark, run_greedy_phase, compliance_report

rows = load_benchmark("data/gsm8k_test_50.jsonl")
assert len(rows) == 50, len(rows)
if PROBLEM_LIMIT:
    rows = rows[:PROBLEM_LIMIT]
print(f"Running on {len(rows)} problems")

t0 = time.time()
greedy_records = run_greedy_phase(rows, generate_fn, "outputs/greedy_checkpoint.jsonl", progress=tqdm)
print(f"Greedy phase: {time.time() - t0:.0f}s")

COMPLIANCE = compliance_report(greedy_records, rows)
print(json.dumps(COMPLIANCE, indent=2))

first = greedy_records[rows[0]["id"]]["greedy"]
print("\n--- Example greedy completion (problem 1) ---\n" + first["completion"])
print("parsed steps:", len(first["steps"]), "| answer:", first["answer"], "| gt:", rows[0]["ground_truth_answer"])
for pid in COMPLIANCE.get("unparsed_ids", [])[:3]:
    print(f"\n--- UNPARSED: {pid} ---\n" + greedy_records[pid]["greedy"]["completion"][-600:])

if COMPLIANCE_GATE is not None and COMPLIANCE["answer_parsed_rate_%"] < COMPLIANCE_GATE:
    raise RuntimeError(
        f"Only {COMPLIANCE['answer_parsed_rate_%']}% of greedy outputs have a parseable Final Answer "
        f"(gate {COMPLIANCE_GATE}%). Fix the prompt in src/generation.py before the long run."
    )

Running on 50 problems


Greedy: 0it [00:00, ?it/s]

Greedy phase: 0s
{
  "num_problems": 50,
  "final_answer_delimiter_rate_%": 100.0,
  "answer_parsed_rate_%": 96.0,
  "has_steps_rate_%": 100.0,
  "mean_steps": 4.16,
  "greedy_accuracy_%": 68.0,
  "drift_counts": {
    "boxed": 0,
    "latex": 19,
    "markdown_heading": 0,
    "bold": 0
  },
  "unparsed_ids": [
    "gsm8k_test_26",
    "gsm8k_test_27"
  ]
}

--- Example greedy completion (problem 1) ---
Step 1: Calculate the total number of eggs laid per day.
Janet's ducks lay 16 eggs per day.

Step 2: Determine how many eggs Janet eats for breakfast each day.
She eats 3 eggs for breakfast every morning.

Step 3: Calculate how many eggs are left after breakfast.
Total eggs - Eggs eaten for breakfast = Remaining eggs

Remaining eggs = 16 - 3 = 13 eggs

Step 4: Subtract the number of eggs used to bake muffins.
She uses 4 eggs to bake muffins for her friends.

Remaining eggs after baking = Remaining eggs - Eggs used for muffins

Remaining eggs after baking = 13 - 4 = 9 eggs

Step 5: Calc

## 8. Phase B — SC@5, PRM judging, PRM Best-of-5 × 3 aggregations, ablation, SC@15

Per problem: 5 shared samples → judge every step once (batched + cached) → PRM rows reuse the same verdicts → 10 extra samples for SC@15. Each finished problem is checkpointed immediately.

In [8]:
from src.pipeline import run_main_phase

t0 = time.time()
records = run_main_phase(rows, generate_fn, judge, greedy_records, "outputs/per_problem.jsonl", progress=tqdm)
MAIN_PHASE_SECONDS = time.time() - t0
print(f"Main phase: {MAIN_PHASE_SECONDS / 60:.1f} min for {len(records)} problems")

Resuming: 50 problems already done, 0 to go.


Main run: 0it [00:00, ?it/s]

Main phase: 0.0 min for 50 problems


## 9. Build the artifacts

In [9]:
from src.pipeline import read_jsonl, build_artifacts, format_table

wanted = {r["id"] for r in rows}
records = [r for r in read_jsonl("outputs/per_problem.jsonl") if r["id"] in wanted]
ART = build_artifacts(records, "outputs")

print(f"Problems evaluated: {ART['num_problems']}")
print("\nresults_summary.json\n" + format_table(ART["summary"]))
print("\naggregation_ablation.json\n" + format_table(ART["ablation"]))
print("\nsc_scaling_curve.json\n" + format_table(ART["curve"]))
print("\nJudge outputs without a 'Verdict:' line:", ART["judge_verdict_line_missing"])

Problems evaluated: 50

results_summary.json
Method                                   | Accuracy (%) | Generation Calls | Judge Calls | Total LLM Calls | Avg Gen Tokens / Problem | Avg Judge Tokens / Problem | Total Time (s)
-----------------------------------------+--------------+------------------+-------------+-----------------+--------------------------+----------------------------+---------------
Greedy (N=1)                             | 68.0         | 50               | 0           | 50              | 202.74                   | 0.0                        | 394.23        
Self-Consistency (N=5)                   | 66.0         | 250              | 0           | 250             | 980.64                   | 0.0                        | 517.63        
Compute-Matched SC (N=15)                | 76.0         | 750              | 0           | 750             | 2931.48                  | 0.0                        | 1218.55       
PRM Best-of-5 (min)                      | 74.0        

## 10. Optional bonus — continuous logprob PRM (`product_logprob`)

Rescores the **same** judged steps of the 5 shared candidates with continuous confidence
p_t = exp(ℓ_VALID) / (exp(ℓ_VALID) + exp(ℓ_INVALID)), read from the judge's logits for the verdict that follows
`judge prompt + "Verdict:"` (every judge output in the main run starts with `Verdict: VALID|INVALID`).
Uses **full-sequence** log-probabilities (teacher forcing over all tokens of " VALID" / " INVALID"); first-token
probabilities are recorded for comparison. Path score = ∏ p_t, then the same filter-then-vote selection.
No text is generated. The four required artifacts are not modified; results go to
`outputs/logprob_bonus_summary.json` and `outputs/logprob_bonus.jsonl`.

In [10]:
BONUS = None
if RUN_LOGPROB_BONUS:
    from src.utils import make_hf_verdict_logprob_fn
    from src.pipeline import CachedLogprob, run_logprob_phase, build_logprob_artifacts

    logprob_fn = make_hf_verdict_logprob_fn(judge_model, judge_tokenizer)
    scorer = CachedLogprob(logprob_fn, batch_size=LOGPROB_BATCH_SIZE)
    n_v = len(judge_tokenizer.encode(" VALID", add_special_tokens=False))
    n_i = len(judge_tokenizer.encode(" INVALID", add_special_tokens=False))
    APPROACH = (
        "Full-sequence log-probability of ' VALID' vs ' INVALID' as the continuation of "
        "chat-templated judge prompt + 'Verdict:' (teacher forcing; first-token probabilities recorded "
        f"for comparison). Qwen2.5 token lengths: ' VALID' = {n_v}, ' INVALID' = {n_i}; "
        f"{logprob_fn.num_sequences_per_prompt} forward sequence(s) per judged step."
    )
    print(APPROACH)
    t0 = time.time()
    bonus_records = run_logprob_phase(records, scorer, "outputs/logprob_bonus.jsonl", progress=tqdm)
    BONUS = build_logprob_artifacts(bonus_records, records, "outputs", approach=APPROACH)
    print(f"\nLogprob bonus: {(time.time() - t0) / 60:.1f} min for {BONUS['num_problems']} problems")
    print("\n" + format_table(BONUS["rows"]))
    print(json.dumps({k: v for k, v in BONUS.items() if k not in ("rows", "approach")}, indent=2))

Full-sequence log-probability of ' VALID' vs ' INVALID' as the continuation of chat-templated judge prompt + 'Verdict:' (teacher forcing; first-token probabilities recorded for comparison). Qwen2.5 token lengths: ' VALID' = 1, ' INVALID' = 1; 1 forward sequence(s) per judged step.


Logprob bonus:   0%|          | 0/50 [00:00<?, ?it/s]


Logprob bonus: 7.0 min for 50 problems

Strategy                         | Accuracy (%) | Tie Rate (%)
---------------------------------+--------------+-------------
product_logprob                  | 80.0         | 0.0         
product_logprob_first_token      | 80.0         | 0.0         
min (binary)                     | 74.0         | 80.0        
product (binary)                 | 76.0         | 42.0        
step_penalty lambda=0.0 (binary) | 78.0         | 68.0        
step_penalty lambda=0.5 (binary) | 78.0         | 68.0        
step_penalty lambda=2.0 (binary) | 76.0         | 68.0        
{
  "num_problems": 50,
  "steps_scored": 1149,
  "verdict_agreement_%": 99.91,
  "mean_p_valid_when_judge_said_VALID": 0.968359,
  "mean_p_valid_when_judge_said_INVALID": 0.068462,
  "steps_with_p_between_0.1_and_0.9": 140,
  "max_abs_diff_full_vs_first_token": 0.0,
  "problems_where_selection_differs_from_binary_product": 18,
  "total_logprob_calls": 1149,
  "total_time_s": 422.36
}


## 11. Spot-check one problem and save run metadata

In [11]:
rec = records[0]
print(f"{rec['id']}  ground truth = {rec['ground_truth_answer']}")
for c in rec["candidates"][:5]:
    print(f"\n[candidate {c['index']}] answer={c['answer']} step_scores={c['step_scores']}")
    for step, out in zip(c["steps"], c["judge_outputs"]):
        print(f"   STEP: {step[:110]}\n   JUDGE: {out.strip()[:160]!r}")
print("\nMethod answers:", {m: v["answer"] for m, v in rec["methods"].items()})

ENV["compliance"] = COMPLIANCE
ENV["judge_verdict_line_missing"] = ART["judge_verdict_line_missing"]
ENV["judge_batch_size"] = JUDGE_BATCH_SIZE
ENV["num_problems"] = ART["num_problems"]
ENV["restored_checkpoints_from_previous_run"] = RESTORED
if BONUS is not None:
    ENV["logprob_bonus"] = {"approach": BONUS["approach"], "total_time_s": BONUS["total_time_s"],
                            "logprob_batch_size": LOGPROB_BATCH_SIZE}
ENV["notebook_wall_clock_minutes_this_session"] = round((time.time() - RUN_START) / 60, 1)
with open("outputs/run_environment.json", "w", encoding="utf-8") as f:
    json.dump(ENV, f, indent=2)
print("\nSaved outputs/run_environment.json")
print(sorted(os.listdir("outputs")))

gsm8k_test_01  ground truth = 18

[candidate 0] answer=18 step_scores=[1.0, 1.0, 1.0, 1.0, 1.0]
   STEP: Calculate the total number of eggs laid by the ducks per day.
Total eggs = 16 eggs/day * 1 day = 16 eggs
   JUDGE: 'Verdict: VALID\nRationale: The calculation correctly multiplies the number of eggs laid per day by the given number of days (1 in this case), which is consistent'
   STEP: Calculate the number of eggs eaten by Janet for breakfast.
Eggs eaten for breakfast = 3 eggs/day * 1 day = 3 e
   JUDGE: 'Verdict: VALID\nRationale: The calculation in the Current Step correctly multiplies the number of eggs eaten for breakfast by the number of days (which is 1) and'
   STEP: Calculate the number of eggs used for baking muffins.
Eggs used for muffins = 4 eggs/day * 1 day = 4 eggs
   JUDGE: 'Verdict: VALID\nRationale: The calculation in the Current Step correctly uses the number of eggs given in the problem statement and the preceding steps to determ'
   STEP: Calculate the number of 

## Done

Download from the notebook's **Output** tab (or `/kaggle/working/pa3/outputs/`):
`per_problem.jsonl`, `results_summary.json`, `aggregation_ablation.json`, `sc_scaling_curve.json` (+ `run_environment.json`, `greedy_checkpoint.jsonl`, and the bonus files `logprob_bonus_summary.json` / `logprob_bonus.jsonl`).